In [1]:
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from matplotlib import animation
from IPython.display import HTML, Video, display
import tqdm as tqdm
from skimage.feature import local_binary_pattern
from scipy.spatial.distance import cosine
import os

In [2]:
df = pd.read_csv("/home/guilherme_monteiro/projetos/tcc/data/metadata/video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "/home/guilherme_monteiro/projetos/tcc/data/videos/" + x)
df_true = df[df['Video Ground Truth'] == "Real"].reset_index(drop=True)
df_fake = df[df['Video Ground Truth'] == "Fake"].reset_index(drop=True)

# Funções básicas

In [3]:
def load_video_frames(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)

    frames = []
    count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frames.append(frame)

        count += 1
        if max_frames and count >= max_frames:
            break

    cap.release()

    return np.array(frames)

def standardize_frame(frame, max_size=640):
    h, w = frame.shape[:2]

    scale = min(max_size / max(h, w), 1.0)
    new_w, new_h = int(w * scale), int(h * scale)

    frame_resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

    return frame_resized, scale

def load_metadata(video_path):
    path = video_path.replace(".mp4", "_meta.json")

    full_path = os.path.join("/home/guilherme_monteiro/projetos/tcc/data/metadata", os.path.basename(path))

    with open(full_path, "r") as f:
        return json.load(f)

def create_face_regions(frame, bbox, padding=0.2):
    h, w = frame.shape[:2]

    x1, y1, x2, y2 = bbox

    bw = x2 - x1
    bh = y2 - y1

    # aplica padding
    px = int(bw * padding)
    py = int(bh * padding)

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(w, x2 + px)
    y2p = min(h, y2 + py)

    # máscara face interna
    face_mask = np.zeros((h, w), dtype=np.uint8)
    face_mask[y1:y2, x1:x2] = 1

    # máscara expandida
    expanded_mask = np.zeros((h, w), dtype=np.uint8)
    expanded_mask[y1p:y2p, x1p:x2p] = 1

    # borda = expandida - interna
    border_mask = expanded_mask - face_mask

    # fundo = resto
    background_mask = 1 - expanded_mask

    return {
        "face": face_mask,
        "border": border_mask,
        "background": background_mask,
        "bbox_expanded": (x1p, y1p, x2p, y2p)
    }


## Função para visualização

In [4]:
def draw_text_block(img, text_lines, x=10, y=20, line_height=18):
    for i, line in enumerate(text_lines):
        cv2.putText(
            img,
            line,
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    return img

# Desenha a caixa delimitadora no frame
def draw_face_box(frame, bbox, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = bbox
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def overlay_regions(frame, regions):
    overlay = frame.copy()

    face_mask = regions["face"]
    border_mask = regions["border"]
    background_mask = regions["background"]

    overlay[face_mask == 1] = (0, 255, 0)       # verde
    overlay[border_mask == 1] = (0, 0, 255)     # vermelho
    overlay[background_mask == 1] = (255, 0, 0) # azul

    alpha = 0.3
    return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

# LBP

In [5]:
def compute_lbp(gray, P=8, R=1):
    lbp = local_binary_pattern(gray, P, R, method="uniform")
    return lbp

In [6]:
def lbp_region_features(lbp, mask):
    p = 8
    r = 1
    n_bins = p * r + 2
    values = lbp[mask == 1]

    if len(values) < 10:
        return None

    hist, _ = np.histogram(values, bins=n_bins, range=(0, n_bins))
    hist = hist.astype(float)
    hist /= (hist.sum() + 1e-6)

    entropy = -np.sum(hist * np.log(hist + 1e-6))
    uniformity = np.sum(hist ** 2)
    sparsity = np.sum(hist > 0.01) / n_bins

    return {
        "hist": hist,
        "entropy": entropy,
        "uniformity": uniformity,
        "sparsity": sparsity
    }

In [7]:
def lbp_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    return {
        f"{prefix}_entropy_diff": abs(f1["entropy"] - f2["entropy"]),
        f"{prefix}_uniformity_diff": abs(f1["uniformity"] - f2["uniformity"]),
        f"{prefix}_sparsity_diff": abs(f1["sparsity"] - f2["sparsity"]),
        f"{prefix}_hist_dist": cosine(f1["hist"], f2["hist"])
    }

In [8]:
def compute_lbp_metrics(frame, bbox):

    frame_std, _ = standardize_frame(frame)
    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

    regions = create_face_regions(frame_std, bbox)

    lbp = compute_lbp(gray)

    face_feat = lbp_region_features(lbp, regions["face"])
    border_feat = lbp_region_features(lbp, regions["border"])
    bg_feat = lbp_region_features(lbp, regions["background"])

    features = {}

    # métricas diretas
    if face_feat:
        features["lbp_face_entropy"] = face_feat["entropy"]
        features["lbp_face_uniformity"] = face_feat["uniformity"]

    if border_feat:
        features["lbp_border_entropy"] = border_feat["entropy"]

    if bg_feat:
        features["lbp_bg_entropy"] = bg_feat["entropy"]

    # diferenças 
    features.update(lbp_region_differences(face_feat, bg_feat, "face_bg"))
    features.update(lbp_region_differences(face_feat, border_feat, "face_border"))
    features.update(lbp_region_differences(border_feat, bg_feat, "border_bg"))

    return features, lbp

## Apenas para visualizar o LBP

In [9]:
def normalize_lbp_for_display(lbp):
    lbp_norm = cv2.normalize(lbp, None, 0, 255, cv2.NORM_MINMAX)
    return lbp_norm.astype(np.uint8)

def debug_lbp_video(video_path, max_frames=500, interval=60):

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    debug_frames = []

    print("\nGerando frames de debug...")

    for i, frame in enumerate(tqdm.tqdm(frames)):

        frame_std, scale = standardize_frame(frame)

        meta = metadata[i]
        bbox = meta["bbox"]
        bbox_scaled = [int(x * scale) for x in bbox]

        # regiões
        regions = create_face_regions(frame_std, bbox_scaled)

        # LBP + features
        features, lbp = compute_lbp_metrics(frame, bbox_scaled)

        lbp_vis = normalize_lbp_for_display(lbp)
        lbp_vis = cv2.cvtColor(lbp_vis, cv2.COLOR_GRAY2BGR)

        # overlay regiões
        overlay = overlay_regions(frame_std.copy(), regions)

        # desenhar bbox
        overlay = draw_face_box(overlay, bbox_scaled)

        # montar texto (apenas principais métricas)
        text_lines = [
            f"Frame: {i}",
            f"Face Entropy: {features.get('lbp_face_entropy', 0):.3f}",
            f"Face Uniform: {features.get('lbp_face_uniformity', 0):.3f}",
            f"Face-BG HistDist: {features.get('face_bg_hist_dist', 0):.3f}",
            f"Face-Border HistDist: {features.get('face_border_hist_dist', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        # concatenar visual
        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(lbp_vis, cv2.COLOR_BGR2RGB)
        ])

        debug_frames.append(combined)

    print("\nDebug pronto. Renderizando...")

    # animação
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")

    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(debug_frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    mpl.rcParams["animation.embed_limit"] = 500000000
    display(HTML(ani.to_jshtml()))

In [10]:
#debug_lbp_video(df_true['video_path'].iloc[0], max_frames=300)

In [11]:
#debug_lbp_video(df_fake['video_path'].iloc[2], max_frames=300)

# Sobel

In [12]:
def compute_sobel(gray):

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

    mag = cv2.magnitude(gx, gy)
    angle = cv2.phase(gx, gy, angleInDegrees=False)

    return mag, angle

In [13]:
def sobel_region_features(mag, angle, mask, n_bins=12):

    mag_vals = mag[mask == 1] # magnitude dos pixels na região, n == 1 para máscara binária que indica os pixels da região
    ang_vals = angle[mask == 1] # ângulo dos pixels na região

    if len(mag_vals) < 20:
        return None

    # hist angular
    hist, _ = np.histogram(ang_vals, bins=n_bins, range=(0, 2*np.pi))
    hist = hist.astype(float)
    hist /= (hist.sum() + 1e-6)

    # entropia angular
    entropy = -np.sum(hist * np.log(hist + 1e-6))

    # coerência circular 
    mean_sin = np.mean(np.sin(ang_vals))
    mean_cos = np.mean(np.cos(ang_vals))
    coherence = np.sqrt(mean_sin**2 + mean_cos**2)

    # energia estrutural (normalizada)
    energy = np.mean(mag_vals)

    return {
        "hist": hist,
        "entropy": entropy,
        "coherence": coherence,
        "energy": energy
    }

In [14]:
def sobel_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    return {
        f"{prefix}_entropy_diff": abs(f1["entropy"] - f2["entropy"]),
        f"{prefix}_coherence_diff": abs(f1["coherence"] - f2["coherence"]),
        f"{prefix}_energy_diff": abs(f1["energy"] - f2["energy"]),
        f"{prefix}_hist_dist": cosine(f1["hist"], f2["hist"])
    }

In [15]:
def compute_sobel_metrics(frame, bbox):

    frame_std, _ = standardize_frame(frame)
    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

    regions = create_face_regions(frame_std, bbox)

    mag, angle = compute_sobel(gray)

    face = sobel_region_features(mag, angle, regions["face"])
    border = sobel_region_features(mag, angle, regions["border"])
    bg = sobel_region_features(mag, angle, regions["background"])

    features = {}

    # diretas
    if face:
        features["sobel_face_entropy"] = face["entropy"]
        features["sobel_face_coherence"] = face["coherence"]

    # derivadas 
    features.update(sobel_region_differences(face, bg, "face_bg"))
    features.update(sobel_region_differences(face, border, "face_border"))
    features.update(sobel_region_differences(border, bg, "border_bg"))

    return features, mag, angle

## Visualização para debug


In [16]:
def sobel_visual(mag, angle):

    mag_vis = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    mag_vis = mag_vis.astype(np.uint8)

    angle_vis = (angle / (2*np.pi) * 255).astype(np.uint8)

    angle_vis = cv2.applyColorMap(angle_vis, cv2.COLORMAP_HSV)

    return mag_vis, angle_vis

def debug_sobel_video(video_path, max_frames=500, interval=60):

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    debug_frames = []

    print("\nProcessando Sobel...")

    for i, frame in enumerate(tqdm.tqdm(frames)):

        frame_std, scale = standardize_frame(frame)

        bbox = metadata[i]["bbox"]
        bbox_scaled = [int(x * scale) for x in bbox]

        regions = create_face_regions(frame_std, bbox_scaled)

        features, mag, angle = compute_sobel_metrics(frame, bbox_scaled)

        mag_vis, angle_vis = sobel_visual(mag, angle)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {i}",
            f"Face Entropy: {features.get('sobel_face_entropy', 0):.3f}",
            f"Face Coherence: {features.get('sobel_face_coherence', 0):.3f}",
            f"Face-BG HistDist: {features.get('face_bg_hist_dist', 0):.3f}",
            f"Face-BG CoherenceDiff: {features.get('face_bg_coherence_diff', 0):.3f}",
            f"Face-BG EnergyDiff: {features.get('face_bg_energy_diff', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(mag_vis, cv2.COLOR_GRAY2RGB),
            cv2.cvtColor(angle_vis, cv2.COLOR_BGR2RGB)
        ])

        debug_frames.append(combined)

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.axis("off")

    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(debug_frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    display(HTML(ani.to_jshtml()))

In [17]:
# debug_sobel_video(df_true['video_path'].iloc[0], max_frames=300)

In [18]:
# debug_sobel_video(df_fake['video_path'].iloc[1], max_frames=200)

# Laplace

In [19]:
def compute_laplacian(gray):
    lap = cv2.Laplacian(gray, cv2.CV_32F, ksize=3)
    return lap

In [20]:
from scipy.stats import kurtosis

def laplacian_region_features(lap, mask, n_bins=20):

    values = lap[mask == 1]

    if len(values) < 20:
        return None

    # normaliza para estabilidade
    values = values / (np.std(values) + 1e-6)

    # histograma
    hist, _ = np.histogram(values, bins=n_bins, range=(-5, 5))
    hist = hist.astype(float)
    hist /= (hist.sum() + 1e-6)

    # variancia, quanto menor o valor, mais estável é o sinal
    energy = np.var(values)

    # kurtosis para picos
    kurt = kurtosis(values)

    # sparsity, quanto do sinal é ativo
    sparsity = np.mean(np.abs(values) > 1.0)

    return {
        "hist": hist,
        "energy": energy,
        "kurtosis": kurt,
        "sparsity": sparsity
    }

In [21]:
def laplacian_region_differences(f1, f2, prefix):
    if f1 is None or f2 is None:
        return {}

    return {
        f"{prefix}_energy_diff": abs(f1["energy"] - f2["energy"]),
        f"{prefix}_kurtosis_diff": abs(f1["kurtosis"] - f2["kurtosis"]),
        f"{prefix}_sparsity_diff": abs(f1["sparsity"] - f2["sparsity"]),
        f"{prefix}_hist_dist": cosine(f1["hist"], f2["hist"])
    }

In [22]:
def compute_laplacian_metrics(frame, bbox):

    frame_std, _ = standardize_frame(frame)
    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

    regions = create_face_regions(frame_std, bbox)

    lap = compute_laplacian(gray)

    face = laplacian_region_features(lap, regions["face"])
    border = laplacian_region_features(lap, regions["border"])
    bg = laplacian_region_features(lap, regions["background"])

    features = {}

    # diretas
    if face:
        features["lap_face_energy"] = face["energy"]
        features["lap_face_kurtosis"] = face["kurtosis"]

    # derivadas
    features.update(laplacian_region_differences(face, bg, "face_bg"))
    features.update(laplacian_region_differences(face, border, "face_border"))
    features.update(laplacian_region_differences(border, bg, "border_bg"))

    return features, lap

## Visualização para debugging

In [23]:
def laplacian_visual(lap):

    lap_abs = np.abs(lap)
    lap_vis = cv2.normalize(lap_abs, None, 0, 255, cv2.NORM_MINMAX)

    return lap_vis.astype(np.uint8)

def debug_laplacian_video(video_path, max_frames=500, interval=60):

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    debug_frames = []

    print("\nProcessando Laplacian...")

    for i, frame in enumerate(tqdm.tqdm(frames)):

        frame_std, scale = standardize_frame(frame)

        bbox = metadata[i]["bbox"]
        bbox_scaled = [int(x * scale) for x in bbox]

        regions = create_face_regions(frame_std, bbox_scaled)

        features, lap = compute_laplacian_metrics(frame, bbox_scaled)

        lap_vis = laplacian_visual(lap)

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {i}",
            f"Face Energy: {features.get('lap_face_energy', 0):.3f}",
            f"Face Kurtosis: {features.get('lap_face_kurtosis', 0):.3f}",
            f"Face-BG HistDist: {features.get('face_bg_hist_dist', 0):.3f}",
            f"Face-BG KurtDiff: {features.get('face_bg_kurtosis_diff', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(lap_vis, cv2.COLOR_GRAY2RGB)
        ])

        debug_frames.append(combined)

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")

    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(debug_frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    display(HTML(ani.to_jshtml()))

In [24]:
# debug_laplacian_video(df_true['video_path'].iloc[1], max_frames=300)

In [25]:
# debug_laplacian_video(df_fake['video_path'].iloc[0], max_frames=300)

# Resumo de metricas

In [ ]:
def all_metrics(video_path, max_frames=500, label=None):
    video_name = os.path.basename(video_path)
    print(f"\nExtraindo métricas do vídeo: {video_name}")

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    all_features = []

    for i, frame in enumerate(frames):

        frame_std, scale = standardize_frame(frame)

        bbox = metadata[i]["bbox"]

        features_lbp, _ = compute_lbp_metrics(frame, bbox)
        features_sobel, _, _ = compute_sobel_metrics(frame, bbox)
        features_lap, _ = compute_laplacian_metrics(frame, bbox)

        features = {**features_lbp, **features_sobel, **features_lap}
        features["frame"] = i

        all_features.append(features)

    df = pd.DataFrame(all_features)
    df.insert(0, "video_name", video_name)
    if label is not None:
        df.insert(1, "label", label)

    return df

In [29]:
df = all_metrics(df['video_path'].iloc[0], label=df['Video Ground Truth'].iloc[0])


Extraindo métricas do vídeo: veo03_velho_cidade.mp4


In [34]:
df.columns

Index(['video_name', 'lbp_face_entropy', 'lbp_face_uniformity',
       'lbp_border_entropy', 'lbp_bg_entropy', 'face_bg_entropy_diff',
       'face_bg_uniformity_diff', 'face_bg_sparsity_diff', 'face_bg_hist_dist',
       'face_border_entropy_diff', 'face_border_uniformity_diff',
       'face_border_sparsity_diff', 'face_border_hist_dist',
       'border_bg_entropy_diff', 'border_bg_uniformity_diff',
       'border_bg_sparsity_diff', 'border_bg_hist_dist', 'sobel_face_entropy',
       'sobel_face_coherence', 'face_bg_coherence_diff', 'face_bg_energy_diff',
       'face_border_coherence_diff', 'face_border_energy_diff',
       'border_bg_coherence_diff', 'border_bg_energy_diff', 'lap_face_energy',
       'lap_face_kurtosis', 'face_bg_kurtosis_diff',
       'face_border_kurtosis_diff', 'border_bg_kurtosis_diff', 'frame',
       'label'],
      dtype='str')